# Games against Stockfish at depths 1 to 7

Plays the final system (Mistral 7B v2, 10 candidates reranked by Stockfish at depth 6) against
Stockfish at depths 1 to 7: 20 games per depth, colours alternating, games longer than 300 plies
counted as draws. These are the games against Stockfish reported in the thesis.

1. **Runtime → Change runtime type → GPU → Save**
2. Run the cells in order.

Each match can take an hour or more. Every finished match is saved to Google Drive, so if the
session stops, run the cells again and the finished matches are skipped.

If Google Drive cannot be mounted, the notebook carries on without it and downloads each finished
match to your computer. If the session stops in that case, remove the finished depths from `DEPTHS`
and run the cells again.

In [ ]:
import os
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/chess_llm_results/games_vs_stockfish")
    ON_DRIVE = True
except Exception as error:
    print(f"Google Drive is not available ({error}).")
    print("Results stay in this session and each finished match is downloaded to your computer.")
    OUT_DIR = Path("/content/games_vs_stockfish")
    ON_DRIVE = False

if not os.path.isdir("/content/chess-llm"):
    !git clone -q https://github.com/Lefterisstam/chess-llm.git /content/chess-llm
%cd /content/chess-llm
!pip install -q -r requirements.txt
!apt-get install -qq stockfish

from model.inference.stockfish_rerank import stockfish_version
print(stockfish_version("/usr/games/stockfish"))

In [ ]:
ADAPTER = "Leftstam/chess-llm-mistral-7b-v2-lora"
STOCKFISH = "/usr/games/stockfish"
DEPTHS = [1, 2, 3, 4, 5, 6, 7]
GAMES = 20

OUT_DIR.mkdir(parents=True, exist_ok=True)

def result_path(depth):
    return OUT_DIR / f"game_sim_llm-stockfish-rerank_vs_stockfish-{depth}.json"

## Play the matches

In [ ]:
from google.colab import files

for depth in DEPTHS:
    out = result_path(depth)
    if out.exists():
        print(f"Depth {depth} is already done, skipping")
        continue
    pgn = out.with_suffix(".pgn")
    !python evaluation/game_simulation.py --agent-a llm:stockfish-rerank --agent-b stockfish:{depth} --adapter {ADAPTER} --stockfish {STOCKFISH} --games {GAMES} --max-plies 300 --rerank-depth 6 --rerank-k 10 --output "{out}" --save-pgn "{pgn}"
    if not ON_DRIVE and out.exists():
        files.download(str(out))
        files.download(str(pgn))

## Results

One line per game: our colour, result, how the game ended, its length, and the material balance
at the end from our side (pawn 1, knight and bishop 3, rook 5, queen 9), followed by the number of
pieces and pawns left on the board.

In [ ]:
import json

import chess

VALUES = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}

def material(board, color):
    return sum(value * len(board.pieces(piece, color)) for piece, value in VALUES.items())

for depth in DEPTHS:
    path = result_path(depth)
    if not path.exists():
        print(f"Depth {depth}: not played yet")
        continue
    d = json.loads(path.read_text(encoding="utf-8"))
    print(f"Stockfish depth {depth}: {d['wins_a']} W / {d['draws']} D / {d['wins_b']} L, "
          f"score {100 * d['score_a']:.1f}% (wins as White {d['wins_a_as_white']}, "
          f"as Black {d['wins_a_as_black']}), {d['config']['stockfish_version']}")
    for i, g in enumerate(d["games"], start=1):
        board = chess.Board(g["final_fen"])
        ours = chess.WHITE if g["white_name"] == d["agent_a"] else chess.BLACK
        balance = material(board, ours) - material(board, not ours)
        print(f"  {i:2d}. {'White' if ours else 'Black'}  {g['result']:8s} {g['termination']:22s} "
              f"{g['n_plies']:4d} plies  material {balance:+3d}  pieces {len(board.piece_map())}")
    print()